# Physics-Motivated Geometric Representation for Propagation-State Prediction in 3D UAV Communication Environments

## Part 2: BaseLine1 Geometric Representation Extraction (Hussein et. al.)

Baseline geometric feature extraction for UAV-to-ground wireless channels.

This module extracts geometric features from UAV/transmitter coordinates,
ground-receiver coordinates, and 3D building models stored in DXF files.

The generated features include:
    - 3D transmitter-receiver distance
    - transmitter height
    - azimuth and elevation angles
    - number of intersected building walls
    - indoor and outdoor path lengths
    - visible-wall count
    - distances to visible walls
    - minimum, maximum, and average intersected-wall heights

The implementation evaluates every UAV-receiver pair for each configured
simulation scenario and stores the resulting feature vectors in CSV files.

The geometric feature definitions correspond to the baseline feature
description provided in the associated research documentation.


## 1. Imports

In [ ]:
import ezdxf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import os
from shapely.geometry import LineString, Polygon, Point


## 2. Experiment Configuration
This section defines the main parameters used for the experiments.

In [ ]:
BASE_DIR = Path(".")
REPRESENTATION_DIR = BASE_DIR / "Inputs_DIR"
OUTPUT_DIR = BASE_DIR / "REPRESENTATION_DIR"

## Dataset-specific input/output configuration.

    Process each configured simulation scenario.

    Each city entry specifies the UAV coordinates, receiver coordinates,
    building geometry, and output file associated with that scenario.
   

In [ ]:
# Simulation scenarios.
# Each scenario defines the UAV/transmitter coordinates, ground-receiver
# coordinates, building DXF geometry, and destination file for the
# generated baseline features.
cities = {
    1: {
        "name": "Seoul",
        "uav_file" : REPRESENTATION_DIR / "Seoul_South Korea_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "Seoul_South Korea_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Seoul_Buildings.dxf",
        "output_file" : OUTPUT_DIR / "Seoul_Hussein_Features.csv",        
    },
    2: {
        "name": "Boston",
        "uav_file" : REPRESENTATION_DIR / "BOSTON_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "BOSTON_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Boston_Building.dxf",
        "output_file" : OUTPUT_DIR / "Boston_Hussein_Features.csv",        
    },
    3: {
        "name": "Pisa_East",
        "uav_file" : REPRESENTATION_DIR / "Pisa_East_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "Pisa_East_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Pisa_East_Building.dxf",
        "output_file" : OUTPUT_DIR / "Pisa_East_Hussein_Features.csv",        
    },
    4: {
        "name": "Pisa_West",
        "uav_file": REPRESENTATION_DIR / "Pisa_West_UAV_Cartesian.csv",
        "gnd_file": REPRESENTATION_DIR / "Pisa_West_GND_Cartesian.csv",
        "dxf_file": REPRESENTATION_DIR / "Pisa_West_Building.dxf",
        "output_file" : OUTPUT_DIR / "Pisa_West_Hussein_Features.csv",        
    },
    5: {
        "name": "London",
        "uav_file": REPRESENTATION_DIR / "London_UAV_Cartesian.csv",
        "gnd_file": REPRESENTATION_DIR / "London_GND_Cartesian.csv",
        "dxf_file": REPRESENTATION_DIR / "London_Building.dxf",
        "output_file" : OUTPUT_DIR / "London_Hussein_Features.csv" ,        
    },
    6: {
        "name": "Manhattan",
        "uav_file" : REPRESENTATION_DIR /  "Manhattan_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR /  "Manhattan_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR /  "Manhattan_Building.dxf",
        "output_file"  : OUTPUT_DIR /  "Manhattan_Hussein_Features.csv" ,        
    },
    7: {
        "name": "Mobile_Seoul1",
        "uav_file" : REPRESENTATION_DIR /  "Mobile_Seoul_South Korea_UAV_TR_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR /  "Mobile_Seoul_South Korea_GND_TR_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR /  "Seoul_Buildings.dxf",
        "output_file" : OUTPUT_DIR /  "Mobile_Seoul_South Korea_TR_Hussein_Features.csv" ,        
    },   
}
   

## Experiment Functions
The following functions implement data loading, and feature extraction.

In [ ]:

def read_tx_rx(uav_file, gnd_file):
    """
    Load UAV and ground-receiver coordinate data from CSV files.

    The input CSV files must contain the Cartesian coordinate columns
    ``Cartesian_X``, ``Cartesian_Y``, and ``Cartesian_Z``. The complete
    DataFrames are returned together with NumPy arrays containing only
    the Cartesian coordinates.

    Parameters
    ----------
    uav_file : str or pathlib.Path
        Path to the CSV file containing UAV/transmitter coordinates.
    gnd_file : str or pathlib.Path
        Path to the CSV file containing ground-receiver coordinates.

    Returns
    -------
    uav_df : pandas.DataFrame
        Complete UAV/transmitter input data.
    gnd_df : pandas.DataFrame
        Complete ground-receiver input data.
    uav_coords : numpy.ndarray
        UAV coordinates with shape ``(N_tx, 3)`` corresponding to
        ``[X, Y, Z]``.
    gnd_coords : numpy.ndarray
        Ground-receiver coordinates with shape ``(N_rx, 3)`` corresponding
        to ``[X, Y, Z]``.

    Notes
    -----
    The coordinate arrays are subsequently used for pairwise geometric
    feature computation between every UAV and ground receiver.
    """
    uav_df = pd.read_csv(uav_file)
    gnd_df = pd.read_csv(gnd_file)    
    uav_coords = uav_df[['Cartesian_X', 'Cartesian_Y', 'Cartesian_Z']].values
    gnd_coords = gnd_df[['Cartesian_X', 'Cartesian_Y', 'Cartesian_Z']].values    
    return uav_df, gnd_df, uav_coords, gnd_coords

def extract_faces_from_dxf(dxf_file):
    """
    Extract 3D building faces from a DXF file.

    The function reads all ``3DFACE`` entities from the DXF model space,
    removes duplicate vertices within each face, and retains faces having
    at least three unique vertices.

    Parameters
    ----------
    dxf_file : str or pathlib.Path
        Path to the DXF file containing the 3D building geometry.

    Returns
    -------
    faces : list of list of tuple
        List of polygonal faces. Each face is represented as a list of
        ``(x, y, z)`` vertex tuples.

    Notes
    -----
    Only DXF ``3DFACE`` entities are considered. Faces with fewer than
    three unique vertices are discarded because they cannot define a
    valid polygon for subsequent geometric processing.
    """
    doc = ezdxf.readfile(dxf_file)
    msp = doc.modelspace()
    faces = []
    for face in msp.query("3DFACE"):
        pts = []
        for i in range(4):
            v = face.dxf.get(f"vtx{i}")
            pts.append((v.x, v.y, v.z))
        # Remove duplicate vertices that may occur in closed or degenerate
        # 3DFACE entities.
        pts_unique = []
        for p in pts:
            if p not in pts_unique:
                pts_unique.append(p)
        # A valid polygon requires at least three unique vertices.
        if len(pts_unique) >= 3:
            faces.append(pts_unique)
    return faces

def extract_vertical_walls(faces):
    """
    Identify vertical building faces and convert them into wall objects.

    Each retained wall is represented by its 2D footprint polygon together
    with its vertical extent. The polygon is used for XY-plane intersection
    and visibility calculations, while the vertical extent is used to
    compute wall-height features.

    Parameters
    ----------
    faces : list of list of tuple
        Building faces extracted from a DXF file. Each vertex is represented
        by an ``(x, y, z)`` tuple.

    Returns
    -------
    walls : list of dict
        List of wall objects. Each dictionary contains:

        ``polygon`` : shapely.geometry.Polygon
            2D XY projection of the wall face.

        ``height`` : float
            Vertical extent of the wall, calculated as ``zmax - zmin``.

        ``zmin`` : float
            Minimum Z-coordinate of the wall.

        ``zmax`` : float
            Maximum Z-coordinate of the wall.

    Notes
    -----
    A face is considered a wall when its vertical extent exceeds 1.0
    coordinate unit. Faces whose XY projection has zero area are discarded.

    The resulting polygons are two-dimensional projections used for the
    line-of-sight and building-intersection calculations.
    """
    walls = []
    for f in faces:
        xs = [p[0] for p in f]
        ys = [p[1] for p in f]
        zs = [p[2] for p in f]
        # Identify faces with a significant vertical extent.
        # These are treated as vertical building walls.
        if np.ptp(zs) > 1.0:
            poly = Polygon([(x,y) for x,y,z in f])
            if poly.area > 0:
                walls.append({
                    "polygon":poly,
                    "height":np.ptp(zs),
                    "zmin":min(zs),
                    "zmax":max(zs)
                    })
    return walls

def wall_intersection_features(tx,rx,walls):
    """
    Compute wall-intersection, building-penetration, and wall-visibility
    features for a transmitter-receiver pair.

    The direct UAV-to-receiver path is projected onto the XY plane.
    Building-wall polygons are then used to determine whether the path
    intersects buildings and which walls are visible from the receiver.

    Parameters
    ----------
    tx : array-like of shape (3,)
        UAV/transmitter Cartesian coordinates ``[x, y, z]``.
    rx : array-like of shape (3,)
        Ground-receiver Cartesian coordinates ``[x, y, z]``.
    walls : list of dict
        Building wall objects produced by :func:`extract_vertical_walls`.
        Each wall must contain ``polygon``, ``height``, ``zmin``, and
        ``zmax`` entries.

    Returns
    -------
    tuple
        The following nine geometric features are returned in order:

        ``wall_pen`` : int
            Number of building walls intersected by the direct
            transmitter-receiver path in the XY plane.

        ``indoor`` : float
            Length of the direct path lying inside intersected building
            polygons in the XY plane.

        ``outdoor`` : float
            Length of the direct path remaining outside building polygons
            in the XY plane.

        ``no_visible_walls`` : int
            Number of walls for which at least one sampled boundary point
            is visible from the receiver without another wall blocking
            the line of sight.

        ``min_dist`` : float
            Minimum distance from the receiver to a visible wall.

        ``avg_dist`` : float
            Average distance from the receiver to all visible walls.

        ``min_h`` : float
            Minimum height of the walls intersected by the direct path.

        ``max_h`` : float
            Maximum height of the walls intersected by the direct path.

        ``avg_h`` : float
            Average height of the walls intersected by the direct path.

    Notes
    -----
    Wall-intersection features are computed using the XY projection of the
    transmitter-receiver path. Therefore, ``indoor`` and ``outdoor`` refer
    to horizontal-plane path lengths rather than full 3D path lengths.

    Wall visibility is estimated by sampling ten points along each wall
    boundary. A wall is considered visible when at least one sampled point
    has an unobstructed line of sight from the receiver.

    The returned feature order is used by
    :func:`extract_all_features_Baseline1`.
    """
    # Construct the direct transmitter-receiver path in the XY plane.
    line = LineString([(tx[0],tx[1]),(rx[0],rx[1])])
    # Initialize path-length statistics.
    # Initially, the complete path is considered outdoor; intersections
    # with building polygons are subsequently transferred to indoor length.
    wall_pen = 0
    indoor = 0
    outdoor = line.length
    intersected_heights=[]
    for wall in walls:
        poly = wall["polygon"]
        # Determine whether the direct path intersects the building footprint.
        if line.intersects(poly):
            inter = line.intersection(poly)
            # Count only intersections having non-zero path length.
            if inter.length > 0:
                wall_pen += 1
                indoor += inter.length
                outdoor -= inter.length
                intersected_heights.append(wall["height"])
    outdoor=max(outdoor,0)
    # Compute minimum, maximum, and average heights of walls intersected
    # by the direct transmitter-receiver path.
    # No intersected walls: assign zero to the wall-height features.
    if len(intersected_heights):
        min_h=np.min(intersected_heights)
        max_h=np.max(intersected_heights)
        avg_h=np.mean(intersected_heights)
    else:
        min_h=0
        max_h=0
        avg_h=0
    # ---------------------------------------------------------
    # Estimate wall visibility from the receiver.
    # ---------------------------------------------------------
    visible_distances = []
    rx_point = Point(rx[0], rx[1])    
    for i, wall in enumerate(walls):    
        # Sample points along the wall boundary
        # Sample a fixed number of points along each wall boundary.
        # A wall is considered visible if at least one sampled point has
        # an unobstructed line of sight from the receiver.
        # Wall visibility is approximated by sampling ten points along each wall boundary.
        boundary = wall["polygon"].boundary
        n_samples = 10    
        visible = False    
        for k in range(n_samples):
            p = boundary.interpolate(k/(n_samples-1), normalized=True)    
            los = LineString([rx_point, p])    
            blocked = False    
            for j, other in enumerate(walls):    
                if i == j:
                    continue    
                # If another building intersects the LOS, the view is blocked
                if los.intersects(other["polygon"]):
                    blocked = True
                    break    
            # At least one unobstructed boundary point is sufficient to classify
            # the wall as visible.
            if not blocked:
                visible = True
                break    
        if visible:
            visible_distances.append(rx_point.distance(wall["polygon"]))
    # Aggregate distances from the receiver to all visible walls.
    if visible_distances:
        no_visible_walls = len(visible_distances)
        min_dist = min(visible_distances)
        avg_dist = np.mean(visible_distances)
    else:
        no_visible_walls = 0
        min_dist = 0
        avg_dist = 0    
    return (wall_pen,indoor,outdoor,no_visible_walls,min_dist,avg_dist,min_h,max_h,avg_h)


def extract_all_features_Baseline1(dxf_file, uav_coords, gnd_coords, output_file):
    """
    Generate the baseline geometric features for all UAV-receiver pairs.

    The function extracts building walls from the DXF file, computes the
    basic transmitter-receiver geometric features, calculates wall-related
    features for every UAV-receiver pair, and incrementally writes the
    resulting records to a CSV file.

    Parameters
    ----------
    dxf_file : str or pathlib.Path
        Path to the DXF file containing the building geometry.

    uav_coords : numpy.ndarray of shape (N_tx, 3)
        Cartesian coordinates of UAV transmitters.

    gnd_coords : numpy.ndarray of shape (N_rx, 3)
        Cartesian coordinates of ground receivers.

    output_file : str or pathlib.Path
        Path to the CSV file used to store the generated features.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing one row for every UAV-receiver pair and
        all computed baseline geometric features.

    Notes
    -----
    The function evaluates all ``N_tx × N_rx`` transmitter-receiver pairs.

    Basic geometric features are computed using
    :func:`compute_basic_features_vectorized`, while wall-related features
    are computed using :func:`wall_intersection_features`.

    Results are appended to ``output_file`` during processing to reduce
    the risk of losing all generated data if processing is interrupted.
    """
    # ---------------------------------------------------------
    # Extract and preprocess building geometry.
    # ---------------------------------------------------------
    faces = extract_faces_from_dxf(dxf_file)
    walls = extract_vertical_walls(faces)
    # Compute features that depend only on transmitter-receiver coordinates.
    df = compute_basic_features_vectorized(uav_coords, gnd_coords)
    # Each UAV is evaluated against every ground receiver.
    n_rx = len(gnd_coords)    
    records = []      
    for tx_idx, tx in enumerate(uav_coords):
        print("TX", tx_idx + 1)
        for rx_idx, rx in enumerate(gnd_coords): 
            # Map the 2D transmitter-receiver indices to the corresponding
            # flattened row index used by the vectorized feature DataFrame.
            idx = tx_idx * n_rx + rx_idx
            wp, ind, out, vis, mind, avgd, minh, maxh, avgh = \
                wall_intersection_features(tx, rx, walls)
            df_record = df.iloc[[idx]].copy()
            # Append wall-intersection and wall-visibility features to the
            # corresponding transmitter-receiver record.
            df_record["Wall_pen"] = wp
            df_record["indoor_distance"] = ind
            df_record["outdoor_distance"] = out
            df_record["no_visible_walls"] = vis
            df_record["min_distance_to_walls"] = mind
            df_record["avg_distance_to_walls"] = avgd
            df_record["min_wall_height"] = minh
            df_record["max_wall_height"] = maxh
            df_record["avg_wall_height"] = avgh
            if not os.path.exists(output_file):
                df_record.to_csv(output_file, index=False)
            else:
                df_record.to_csv(output_file,mode="a",header=False,index=False)
            records.append(df_record)

    df_final = pd.concat(records, ignore_index=True)

    return df_final



def compute_basic_features_vectorized(uav_coords, gnd_coords):
    """
    Compute basic geometric features for every UAV-receiver pair.

    The computation is vectorized over all transmitter and receiver
    coordinates, avoiding an explicit Python loop over individual pairs.

    Parameters
    ----------
    uav_coords : numpy.ndarray of shape (N_tx, 3)
        Cartesian coordinates of UAV transmitters in the form
        ``[x, y, z]``.

    gnd_coords : numpy.ndarray of shape (N_rx, 3)
        Cartesian coordinates of ground receivers in the form
        ``[x, y, z]``.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing ``N_tx * N_rx`` transmitter-receiver records
        with the following features:

        ``TX`` : int
            One-based transmitter index.

        ``RX`` : int
            One-based receiver index.

        ``3d_distance`` : float
            Euclidean distance between the UAV and receiver.

        ``Transmitte_ZZ`` : float
            UAV height (Z-coordinate).

        ``Theta`` : float
            Azimuth angle of the transmitter-receiver vector in the XY
            plane, measured from the positive X-axis in degrees and
            normalized to [0, 360).

        ``Phi`` : float
            Elevation angle of the transmitter-receiver vector relative
            to the XY plane, in degrees.

    Notes
    -----
    The output ordering is transmitter-major: all receivers associated
    with transmitter 1 appear first, followed by all receivers associated
    with transmitter 2, and so on.

    This ordering must remain consistent with the indexing used in
    :func:`extract_all_features_Baseline1`.
    """
    # Construct all pairwise UAV-to-receiver coordinate differences.
    # Resulting shape: (N_tx, N_rx, 3).
    diff = uav_coords[:, None, :] - gnd_coords[None, :, :]
    dx = diff[:, :, 0]
    dy = diff[:, :, 1]
    dz = diff[:, :, 2]
    # Euclidean 3D distance and horizontal (XY-plane) distance.
    distance_3d = np.sqrt(dx**2 + dy**2 + dz**2)
    horizontal = np.sqrt(dx**2 + dy**2)
   # Azimuth angle in the XY plane, measured counter-clockwise from +X.
    theta = np.degrees(np.arctan2(dy, dx))
    theta = np.mod(theta, 360)
    # # Elevation angle relative to the XY plane. Phi
    phi = np.degrees(np.arctan2(dz, horizontal))
    # UAV height
    transmitter_zz = uav_coords[:, 2][:, None]
    n_tx = len(uav_coords)
    n_rx = len(gnd_coords)
    return pd.DataFrame({
        "TX": np.repeat(np.arange(1, n_tx + 1), n_rx),
        "RX": np.tile(np.arange(1, n_rx + 1), n_tx),        
        "3d_distance": distance_3d.ravel(),
        "Transmitte_ZZ": np.repeat(transmitter_zz.ravel(), n_rx),
        "Theta": theta.ravel(),
        "Phi": phi.ravel(),
    })



In [ ]:
## Run Experiments
The following cells execute the experiments for the Baseline1 representation.

In [ ]:
PLOT_DEBUG = False
# Process each configured simulation scenario independently.
for idx, city in cities.items():
    # Retrieve input and output paths for the current scenario.
    name = city["name"]
    uav_file = city["uav_file"]
    gnd_file = city["gnd_file"]
    dxf_file = city["dxf_file"]

    output_file = city["output_file"]
    

    print(f"\nProcessing city {idx}: {name}")
    print("UAV:", uav_file)
    print("GND:", gnd_file)
    print("DXF:", dxf_file)

    # ---------------------------------------------------------
    # 1. Read UAV & GND coordinates
    # ---------------------------------------------------------
    uav_df, gnd_df, uav_coords, gnd_coords = read_tx_rx(uav_file,gnd_file)

    print("Loaded UAV:",uav_coords.shape,"Loaded GND:",gnd_coords.shape)
    # ---------------------------------------------------------
    # 2. Extract features
    # ---------------------------------------------------------
    # Example: process the first UAV/GND pair
    df = extract_all_features_Baseline1(dxf_file,uav_coords,gnd_coords, output_file)
    # ---------------------------------------------------------
    # 3. Plot
    # ---------------------------------------------------------
    if PLOT_DEBUG:
        plot_dxf_with_first_tx_rx(dxf_file,uav_coords,gnd_coords)

    
